# 01 · Data export and preprocessing

Builds a **global** training dataset for Sentinel-1-only LULC segmentation.

| Step | What happens |
|---|---|
| 1 | 40 study areas (20.48 × 20.48 km) on six continents, split by region into train / val / test |
| 2 | Sentinel-1 coverage per orbit pass → one pass per area |
| 3 | Pilot: export and inspect one area for one month |
| 4 | Export all area-months from Google Earth Engine (S1 composite + Dynamic World label) |
| 5 | Integrity check of the downloaded tiles |
| 6 | Cut tiles into 256 × 256 patches, class distribution, normalisation statistics |

**Time split:** training and validation use **2023–2024**, testing uses **2025–2026**.

| Split | Study areas | Period | Purpose |
|---|---|---|---|
| `train` | 27 train areas | 2023–24 | fitting |
| `val` | 4 val areas | 2023–24 | model selection, early stopping |
| `test` | 9 test areas | 2025–26 | **main test**: unseen places, future dates |
| `test_temporal` | 27 train areas | 2025–26 | robustness over time only (places are known) |

**Run on the lab server.** Section 4 can also run in the background with
`nohup python scripts/run_export.py > export.log 2>&1 &`; then the export cell here only verifies the files.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from tqdm.auto import tqdm
import ee
import folium
from pyproj import Transformer

from src import config as C, gee_data as G, patching as P

G.init()
CMAP = ListedColormap(C.CLASS_COLORS)
SPLIT_COLORS = {"train": "#2563eb", "val": "#f59e0b", "test": "#dc2626"}
print("GEE project:", C.GEE_PROJECT, "| raw:", C.RAW_DIR, "| patches:", C.PATCH_DIR)

## 1. Classes and study areas

Eight classes, identical to the **Esri 10 m Annual LULC** map (without its *clouds* class), which is the
independent reference for the prototype case study. Labels come from **Dynamic World** (Sentinel-2 based,
9 classes); grass and shrub/scrub are merged into *rangeland* because their C-band backscatter is too similar.

In [ ]:
pd.DataFrame({"class": C.CLASSES,
              "from Dynamic World": [", ".join(d for d, c in zip(C.DW_BANDS, C.DW_TO_CLASS) if c == i)
                                     for i in range(len(C.CLASSES))]}).rename_axis("index")

In [ ]:
aois = pd.DataFrame(C.AOIS).T.rename_axis("aoi").reset_index()
display(aois)
aois.groupby(["kind", "split"]).size().unstack(fill_value=0)

In [ ]:
m = folium.Map(location=[20, 10], zoom_start=2, tiles="CartoDB positron")
size = C.AOI_PX * C.PIXEL_SIZE
for name, a in C.AOIS.items():
    epsg, x0, y0 = G.aoi_grid(a["lat"], a["lon"])
    to_ll = Transformer.from_crs(epsg, 4326, always_xy=True)
    lon0, lat0 = to_ll.transform(x0, y0 - size)
    lon1, lat1 = to_ll.transform(x0 + size, y0)
    folium.Rectangle([[lat0, lon0], [lat1, lon1]], color=SPLIT_COLORS[a["split"]], weight=3, fill=True,
                     fill_opacity=0.5, tooltip=f"{name} · {a['kind']} · {a['split']}").add_to(m)
    folium.CircleMarker([a["lat"], a["lon"]], radius=4, color=SPLIT_COLORS[a["split"]], fill=True,
                        tooltip=f"{name} ({a['lat']}, {a['lon']}) · {a['kind']} · {a['split']}").add_to(m)
m

## 2. Sentinel-1 coverage and orbit pass

Backscatter depends on look direction and incidence angle, so ascending and descending scenes of the same place
differ. Each study area therefore uses **one pass only**: the one with more scenes in its period.

Since Sentinel-1B failed (December 2021), ESA no longer acquires IW data over some remote regions. Four
originally planned areas (Sahara, Okavango, Gobi, Kazakh steppe) had **zero** scenes in 2023–24 and were replaced
by areas of the same landscape type. Sentinel-1C has been operational since 2025, so the test period has
more scenes per month than the training period.

In [ ]:
rows = []
for name, a in tqdm(C.AOIS.items(), total=len(C.AOIS)):
    months = C.TEST_MONTHS if a["split"] == "test" else C.TRAIN_MONTHS
    start, end = ee.Date(f"{months[0]}-01"), ee.Date(f"{months[-1]}-01").advance(1, "month")
    epsg, x0, y0 = G.aoi_grid(a["lat"], a["lon"])
    geom = G.grid_geometry(epsg, x0, y0, C.AOI_PX, C.AOI_PX)
    counts = ee.Dictionary({o: G.s1_collection(geom, start, end, o).size() for o in ["ASCENDING", "DESCENDING"]}
                           ).set("DynamicWorld", G.dw_collection(geom, start, end).size()).getInfo()
    rows.append(dict(aoi=name, split=a["split"], period=f"{months[0]} – {months[-1]}", **counts, chosen=a["orbit"]))
coverage = pd.DataFrame(rows).set_index("aoi")
coverage

## 3. Pilot: one area, one month

Exports the four 1024 × 1024 tiles of the Everglades for July 2023 and shows the three input channels next to the
label. White label pixels are *ignored* in training: Dynamic World was not confident enough (mean top-class
probability < 0.3) or had no cloud-free observation that month.

In [ ]:
pilot = G.export_aoi_month("everglades_us", C.AOIS["everglades_us"], "2023-07")
pilot

In [ ]:
def show_tile(path):
    s1, label = P.load_tile(path)
    fig, ax = plt.subplots(1, 4, figsize=(20, 5.5))
    for i, (band, (lo, hi)) in enumerate(zip(C.S1_BANDS, C.CLIP_DB)):
        im = ax[i].imshow(s1[..., i], cmap="gray", vmin=lo, vmax=hi)
        ax[i].set_title(f"{band} (dB)")
        plt.colorbar(im, ax=ax[i], fraction=0.046)
    ax[3].imshow(np.ma.masked_equal(label, C.IGNORE), cmap=CMAP, vmin=0, vmax=len(C.CLASSES) - 1,
                 interpolation="nearest")
    ax[3].set_title("Label (Dynamic World → 8 classes)")
    ax[3].legend([plt.Rectangle((0, 0), 1, 1, color=c) for c in C.CLASS_COLORS], C.CLASSES,
                 loc="lower left", fontsize=8, framealpha=0.9)
    for a in ax:
        a.axis("off")
    fig.suptitle(path.stem)
    plt.tight_layout()
    plt.show()

show_tile(C.RAW_DIR / "everglades_us" / "everglades_us_2023-07_t00.tif")

## 4. Full export

Every second month, offset per year so all 12 calendar months are represented: 12 months for train/val
(2023–24) and 11 months for test (2025–26), plus 6 months of 2025–26 for the training areas (`test_temporal`).
Area-months without any Sentinel-1 scene or Dynamic World observation are skipped (e.g. persistent cloud).
Watch the EECU usage on the Earth Engine **Configuration** page (Community tier: 150 EECU-hours per month).

In [ ]:
jobs = C.export_jobs()
print(len(jobs), "area-months,", len(jobs) * (C.AOI_PX // C.TILE_PX) ** 2, "tiles")

logs = []
with ThreadPoolExecutor(max_workers=4) as pool:
    futures = {pool.submit(G.export_aoi_month, n, a, m): (n, m) for n, a, m in jobs}
    for f in tqdm(as_completed(futures), total=len(futures)):
        try:
            logs.append(f.result())
        except Exception as e:
            n, m = futures[f]
            logs.append(dict(aoi=n, month=m, status=f"error: {e}"))

logs = pd.DataFrame(logs).sort_values(["aoi", "month"])
logs.to_csv(C.DATA_DIR / "export_log.csv", index=False)
logs.status.value_counts()

In [ ]:
print("Skipped area-months:")
logs[logs.status != "ok"][["aoi", "month", "s1", "dw", "status"]]

In [ ]:
print("Sentinel-1 scenes per monthly composite:")
logs.pivot(index="aoi", columns="month", values="s1")

In [ ]:
print("Dynamic World observations per label month:")
logs.pivot(index="aoi", columns="month", values="dw")

## 5. Integrity check

An interrupted download can leave an unreadable file. Such tiles are listed here; delete them with
`python scripts/check_tiles.py --delete` and re-run the export.

In [ ]:
bad = P.find_unreadable()
print(len(list(C.RAW_DIR.rglob("*.tif"))), "tiles,", len(bad), "unreadable")
bad

## 6. Patching, splits and class distribution

* Each tile is cut into non-overlapping 256 × 256 patches.
* A patch is dropped if any S1 pixel is missing, or if fewer than 50 % of its pixels have a reliable label.
* Arrays are stored in dB (float16); clipping and normalisation happen in the training pipeline using
  `stats.json`, computed on the **training split only**.

In [ ]:
metas = {s: P.build_split(s) for s in C.SPLITS}
pd.DataFrame({s: {"areas": m.aoi.nunique(), "months": m.month.nunique(), "patches": len(m)}
              for s, m in metas.items()}).T

In [ ]:
pd.concat(metas, names=["split"]).reset_index(0).groupby(["aoi", "split"]).size().unstack(fill_value=0)

In [ ]:
dist = pd.concat({s: P.class_distribution(s).set_index("class")["percent"] for s in metas}, axis=1)
display(dist.round(2))

ax = dist.plot.bar(figsize=(12, 4.5), color=["#2563eb", "#f59e0b", "#dc2626", "#9333ea"], rot=30)
ax.set_ylabel("% of labelled pixels")
ax.set_title("Class distribution per split")
ax.set_yscale("log")
plt.tight_layout()
plt.show()

In [ ]:
stats = P.normalisation_stats()
stats

In [ ]:
X = np.load(C.PATCH_DIR / "X_train.npy", mmap_mode="r")
y = np.load(C.PATCH_DIR / "y_train.npy", mmap_mode="r")
meta = metas["train"]
idx = np.random.default_rng(0).choice(len(X), 8, replace=False)

fig, ax = plt.subplots(2, 8, figsize=(24, 6.5))
for j, i in enumerate(idx):
    ax[0, j].imshow(X[i, ..., 0].astype(np.float32), cmap="gray", vmin=-25, vmax=0)
    ax[0, j].set_title(f"{meta.aoi[i]}\n{meta.month[i]} · VV", fontsize=9)
    ax[1, j].imshow(np.ma.masked_equal(y[i], C.IGNORE), cmap=CMAP, vmin=0, vmax=len(C.CLASSES) - 1,
                    interpolation="nearest")
for a in ax.ravel():
    a.axis("off")
fig.legend([plt.Rectangle((0, 0), 1, 1, color=c) for c in C.CLASS_COLORS], C.CLASSES, loc="lower center",
           ncol=len(C.CLASSES))
plt.tight_layout(rect=(0, 0.06, 1, 1))
plt.show()

## Preprocessing summary (for the report)

| Step | Where | Detail |
|---|---|---|
| Thermal noise removal, radiometric calibration (σ⁰), terrain correction (SRTM 30 m), dB conversion | GEE `COPERNICUS/S1_GRD` | Applied by GEE with the ESA SNAP toolbox |
| Scene selection | `s1_collection` | IW mode, VV+VH, 10 m GRD, **one orbit pass per study area** |
| Border noise | `_prep_scene` | VV < −30 dB masked |
| Speckle filtering | `lee_filter` | Lee filter, 5 × 5 window, ENL 4.4, applied in linear power per scene |
| Monthly compositing | `s1_monthly` | Per-pixel median of all filtered scenes in the month (also temporal multi-looking), back to dB; VV−VH ratio band added |
| Label matching | `dw_monthly` | Same month, same 10 m local-UTM grid: mean of Dynamic World class probabilities over cloud-free observations → argmax |
| Class mapping | `patching.load_tile` | 9 Dynamic World classes → 8 classes (grass + shrub/scrub → rangeland) |
| Label quality | `patching.load_tile` | Pixels with mean top-class probability < 0.3 or no DW observation → ignore index 255 |
| Patching | `build_split` | 256 × 256, non-overlapping; patches with S1 gaps or < 50 % labelled pixels dropped |
| Normalisation | `normalisation_stats` | Clip VV [−25, 0], VH [−32, −5], VV−VH [−5, 20] dB, then z-score with training-split mean/std |
| Splits | `config.SPLITS` | By region **and** time: train/val 2023–24, test (unseen regions) 2025–26 |
| Augmentation | notebook 02 | Random flips and 90° rotations (geometry only, no intensity changes) |